# Interpretability — SHAP, XPER, permutation importance, surrogate, PDP/ICE, LIME

Runs for each model with a `models/<name>_model.py` file (`report_status()` below shows
which). The last cell is a throwaway smoke test for when no real model is in yet — it
skips itself once one is, so it's safe to leave in.

In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path("../..").resolve()))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shap
from sklearn.metrics import roc_auc_score

from common_metrics import (
    FEATURES, MODEL_NAMES, TEAM_THRESHOLD,
    load_split, get_X_y, available_models, report_status,
    manual_permutation_importance, get_or_fit_model, run_step,
)

report_status()

## Load data

In [ ]:
train_df = load_split("train")
test_df = load_split("test")

X_train, y_train = get_X_y(train_df)
X_test, y_test = get_X_y(test_df)

print(f"train: {X_train.shape}, test: {X_test.shape}")

## SHAP — per-prediction attribution

TreeExplainer for XGBoost, LinearExplainer for logreg, KernelExplainer for TabPFN, falling
back to KernelExplainer if the native explainer can't be built from whatever `fit()`
returned (e.g. a pipeline-wrapped model).

In [ ]:
MODEL_EXPLAINER_TYPE = {"xgboost": "tree", "logreg": "linear", "tabpfn": "kernel"}

def get_shap_explainer(model_type, model, module, X_background):
    predict_fn = lambda X: module.predict_proba(model, pd.DataFrame(X, columns=X_background.columns))[:, 1]
    if model_type == "tree":
        probe = X_background.iloc[:2]
        for candidate in (model, getattr(model, "booster", None)):
            if candidate is None:
                continue
            try:
                explainer = shap.TreeExplainer(candidate)
                explainer(probe)  # smoke-test: a wrapper wanting its own preprocessing fails here, not later
                note = "TreeExplainer (exact)" if candidate is model else (
                    "TreeExplainer (exact, on model.booster — pre-calibration score, "
                    "not the final calibrated probability)"
                )
                return explainer, note
            except Exception:
                continue
        print("  TreeExplainer failed on both the model and model.booster — falling back to KernelExplainer")
    elif model_type == "linear":
        try:
            return shap.LinearExplainer(model, X_background), "LinearExplainer (exact)"
        except Exception as e:
            print(f"  LinearExplainer failed ({e}) — falling back to KernelExplainer")
    background_sample = shap.sample(X_background, min(100, len(X_background)))
    return shap.KernelExplainer(predict_fn, background_sample), "KernelExplainer (approximate)"

## XPER — performance decomposition

`pip install XPER` ([github.com/hi-paris/XPER](https://github.com/hi-paris/XPER)). Unlike
SHAP, XPER decomposes a *performance metric* into per-feature Shapley contributions, not a
single prediction. `metric` is a fixed name (`"AUC"`, `"Accuracy"`, `"MC"` for
misclassification cost) rather than an arbitrary function; `"MC"` takes `cfp`/`cfn` for an
economic-cost decomposition. `kernel=True` since the docs recommend it above ~10 features
and we have ~28.

Runtime scales with feature count, not row count — Shapley coalition sampling over ~28
features is slow even in kernel mode. Budget for it, or narrow to fewer columns first.

In [ ]:
try:
    from XPER.compute.Performance import ModelPerformance
    HAS_XPER = True
except ImportError:
    HAS_XPER = False
    print("XPER not installed — pip install XPER, then re-run this cell")


class _XPERModelAdapter:
    """XPER calls model.predict_proba(X) directly; this routes that through our
    module.predict_proba(model, X) contract so it works regardless of what
    fit() returned."""

    def __init__(self, model, module):
        self._model, self._module = model, module

    def predict_proba(self, X):
        return self._module.predict_proba(self._model, X)


def compute_xper(model, module, X_train, y_train, X_test, y_test, metric="AUC", cfp=None, cfn=None):
    if not HAS_XPER:
        return None
    adapter = _XPERModelAdapter(model, module)
    xper = ModelPerformance(X_train, y_train, X_test, y_test, adapter)
    performance = xper.evaluate([metric], CFP=cfp, CFN=cfn)
    phi, phi_i_j = xper.calculate_XPER_values([metric], CFP=cfp, CFN=cfn, kernel=True)
    return {"performance": performance, "phi": phi, "phi_i_j": phi_i_j}

## Permutation importance — cheap cross-check

In [ ]:
def permutation_importance_report(model, module, X, y, n_repeats=10):
    return manual_permutation_importance(model, module, X, y, n_repeats=n_repeats)

## Global surrogate tree — fidelity-checked approximation

A shallow tree fit to reproduce the model's own predicted probabilities, not the true
labels. Always report fidelity (R² against the real model's output) alongside it — a
surrogate is only useful if you know how far it actually is from the model it's standing in for.

In [ ]:
from sklearn.tree import DecisionTreeRegressor, plot_tree
from sklearn.metrics import r2_score

def encode_for_tree(X):
    Xe = X.copy()
    for col in Xe.columns:
        if not pd.api.types.is_numeric_dtype(Xe[col]):
            Xe[col] = Xe[col].astype("category").cat.codes
    return Xe.fillna(-1)


def global_surrogate(model, module, X, max_depth=4):
    probs = module.predict_proba(model, X)[:, 1]
    Xe = encode_for_tree(X)
    surrogate = DecisionTreeRegressor(max_depth=max_depth, random_state=42).fit(Xe, probs)
    fidelity = r2_score(probs, surrogate.predict(Xe))
    return surrogate, fidelity, Xe.columns


def run_surrogate(model, module, X_test, model_name):
    surrogate_sample = X_test.sample(min(5000, len(X_test)), random_state=42)
    surrogate, fidelity, cols = global_surrogate(model, module, surrogate_sample)
    print(f"    surrogate fidelity (R² vs. the real model) = {fidelity:.3f}")
    fig, ax = plt.subplots(figsize=(14, 6))
    plot_tree(surrogate, feature_names=list(cols), max_depth=3, filled=True, fontsize=7, ax=ax)
    ax.set_title(f"{model_name} — global surrogate (depth-limited to 3; fidelity R²={fidelity:.3f})")
    plt.tight_layout()
    plt.show()
    return fidelity

## PDP + ICE — shape of the relationship, not just attribution

SHAP says what matters; this says what the relationship looks like — monotonic, a hard
threshold, does it vary a lot person-to-person (that's what ICE adds over the PDP average).
Scoped to the same three features used for FPDP (`debt_to_income_ratio`,
`combined_loan_to_value_ratio`, `income`) instead of sweeping all of them.

In [ ]:
PDP_FEATURES = [f for f in ["debt_to_income_ratio", "combined_loan_to_value_ratio", "income"] if f in FEATURES]

def compute_pdp_ice(model, module, X, feature, n_points=20, ice_sample_size=50, random_state=42):
    grid = np.linspace(X[feature].quantile(0.05), X[feature].quantile(0.95), n_points)
    rng = np.random.RandomState(random_state)
    ice_idx = rng.choice(len(X), size=min(ice_sample_size, len(X)), replace=False)
    X_ice = X.iloc[ice_idx].reset_index(drop=True)
    ice_curves = np.zeros((len(X_ice), n_points))
    for j, val in enumerate(grid):
        X_mod = X_ice.copy()
        X_mod[feature] = val
        ice_curves[:, j] = module.predict_proba(model, X_mod)[:, 1]
    return grid, ice_curves.mean(axis=0), ice_curves


def plot_pdp_ice(grid, pdp_curve, ice_curves, feature, model_name):
    fig, ax = plt.subplots(figsize=(6, 4))
    for row in ice_curves:
        ax.plot(grid, row, color="steelblue", alpha=0.1, linewidth=0.8)
    ax.plot(grid, pdp_curve, color="black", linewidth=2, label="PDP (average)")
    ax.set_xlabel(feature)
    ax.set_ylabel("P(approved)")
    ax.set_title(f"{model_name} — PDP + ICE — {feature}")
    ax.legend()
    plt.tight_layout()
    plt.show()


def run_pdp_ice_feature(model, module, X_test, feature, model_name):
    pdp_sample = X_test.sample(min(2000, len(X_test)), random_state=42)
    grid, pdp_curve, ice_curves = compute_pdp_ice(model, module, pdp_sample, feature)
    plot_pdp_ice(grid, pdp_curve, ice_curves, feature, model_name)
    return {"grid": grid, "pdp_curve": pdp_curve}

## LIME — on the same representative applicants as SHAP

LIME perturbs every feature it's given with Gaussian noise, which breaks HMDA's
integer-coded categoricals (loan type, lien status, etc.) — a perturbed "2.7" isn't a valid
category. So it only perturbs the genuinely continuous features (income, loan amount, DTI,
LTV, property value, loan term) and holds everything else at the instance's real value.
Run on the same instances as SHAP so the two can be compared directly — this is the
"Disagreement in XAI" check from the syllabus (Krishna et al. 2025), not a SHAP replacement.

In [ ]:
import lime.lime_tabular

LIME_CONTINUOUS_COLS = [f for f in [
    "income", "loan_amount", "debt_to_income_ratio", "combined_loan_to_value_ratio",
    "property_value", "loan_term", "intro_rate_period", "total_units",
] if f in FEATURES]
LIME_OTHER_COLS = [f for f in FEATURES if f not in LIME_CONTINUOUS_COLS]


def get_lime_explainer(X_train):
    Xc = X_train[LIME_CONTINUOUS_COLS].fillna(X_train[LIME_CONTINUOUS_COLS].median())
    explainer = lime.lime_tabular.LimeTabularExplainer(
        Xc.values, feature_names=LIME_CONTINUOUS_COLS, class_names=["denied", "approved"],
        mode="classification", random_state=42,
    )
    return explainer, Xc.median()


def lime_explain(explainer, model, module, instance_row, medians, num_features=8):
    def predict_fn(arr):
        df = pd.DataFrame(arr, columns=LIME_CONTINUOUS_COLS)
        for c in LIME_OTHER_COLS:
            df[c] = instance_row[c]
        return module.predict_proba(model, df[FEATURES])
    row_values = instance_row[LIME_CONTINUOUS_COLS].fillna(medians).values
    return explainer.explain_instance(row_values, predict_fn, num_features=num_features)


def pick_representative_instances(model, module, X):
    probs = module.predict_proba(model, X)[:, 1]
    return {
        "approved (highest score)": int(np.argmax(probs)),
        "denied (lowest score)": int(np.argmin(probs)),
        "borderline (closest to threshold)": int(np.argmin(np.abs(probs - TEAM_THRESHOLD))),
    }

## Run across available models

One loop per model. Getting the model is the only hard dependency — if that fails, skip
the model entirely. Every metric after that (SHAP, XPER, permutation importance, surrogate,
each PDP/ICE feature, each LIME instance) goes through `run_step`, so one failing doesn't
take the rest down with it.

`INTERP_SAMPLE_SIZE` caps AUC/XPER/permutation importance to a subsample for TabPFN only —
permutation importance alone calls `predict_proba` ~156 times (1 baseline + 5 repeats × 31
features), which at TabPFN's per-row inference cost is intractable over the full ~1.37M-row
test set on CPU, and not free even on a shared/contended GPU. SHAP, the surrogate, and PDP/ICE
already sample internally, so they're untouched by this.

In [ ]:
def compute_auc(model, module, X_test, y_test):
    probs = module.predict_proba(model, X_test)[:, 1]
    return roc_auc_score(y_test, probs)


def run_shap(model, module, model_type, X_train, X_test):
    explainer, method = get_shap_explainer(model_type, model, module, X_train)
    test_sample = X_test.sample(min(200, len(X_test)), random_state=42)
    shap_values = explainer(test_sample) if model_type != "kernel" else explainer.shap_values(test_sample)
    print(f"    SHAP method: {method}")
    return shap_values


def run_lime_instance(model, module, explainer, medians, row, num_features=8):
    return lime_explain(explainer, model, module, row, medians, num_features=num_features)


INTERP_SAMPLE_SIZE = {"tabpfn": 5000}  # None (full data) for xgboost/logreg

results = {}

for name, module in available_models().items():
    print(f"\n=== {name} ===")
    try:
        model = get_or_fit_model(name, module, X_train, y_train)
    except Exception as e:
        print(f"  model unavailable ({type(e).__name__}: {e}) — skipping {name} entirely")
        continue

    entry = {"model": model}
    model_type = MODEL_EXPLAINER_TYPE[name]

    sample_size = INTERP_SAMPLE_SIZE.get(name)
    X_train_s, y_train_s, X_test_s, y_test_s = X_train, y_train, X_test, y_test
    if sample_size is not None:
        X_train_s = X_train.sample(min(sample_size, len(X_train)), random_state=42)
        y_train_s = y_train.loc[X_train_s.index]
        X_test_s = X_test.sample(min(sample_size, len(X_test)), random_state=42)
        y_test_s = y_test.loc[X_test_s.index]
        print(f"  AUC/XPER/permutation importance use a {len(X_test_s)}-row subsample for this model")

    if run_step(entry, "auc", compute_auc, model, module, X_test_s, y_test_s):
        print(f"  AUC: {entry['auc']:.4f}")

    run_step(entry, "shap_values", run_shap, model, module, model_type, X_train, X_test)
    run_step(entry, "xper", compute_xper, model, module, X_train_s, y_train_s, X_test_s, y_test_s, metric="AUC")
    run_step(entry, "permutation_importance", permutation_importance_report, model, module, X_test_s, y_test_s, n_repeats=5)
    run_step(entry, "surrogate_fidelity", run_surrogate, model, module, X_test, name)

    entry["pdp_ice"] = {}
    for feature in PDP_FEATURES:
        run_step(entry["pdp_ice"], feature, run_pdp_ice_feature, model, module, X_test, feature, name)

    entry["lime"] = {}
    lime_explainer, lime_medians = get_lime_explainer(X_train)
    for label, idx in pick_representative_instances(model, module, X_test).items():
        row = X_test.iloc[idx]
        if run_step(entry["lime"], label, run_lime_instance, model, module, lime_explainer, lime_medians, row):
            print(f"    LIME {label}: {entry['lime'][label].as_list()[:2]}")

    results[name] = entry

if not results:
    print("No models ready yet — drop a models/<name>_model.py file in and re-run.")

## Global comparison — mean |SHAP| per feature, across models

In [ ]:
if results:
    fig, ax = plt.subplots(figsize=(8, 5))
    for name, r in results.items():
        if "shap_values" not in r:
            continue
        sv = r["shap_values"]
        vals = sv.values if hasattr(sv, "values") else np.asarray(sv)
        mean_abs = np.abs(vals).mean(axis=0)
        ax.barh(FEATURES, mean_abs, alpha=0.5, label=name)
    ax.set_xlabel("mean |SHAP value|")
    ax.legend()
    plt.tight_layout()
    plt.show()

## Summary table — one row per model, copy-ready for the slide deck

A blank (`—`) means that metric failed for that model, not that the row is missing. TabPFN's
AUC here is on the `INTERP_SAMPLE_SIZE` subsample, not the full test set — not directly
comparable to xgboost/logreg's full-test-set AUC without noting that in the write-up.

In [ ]:
def top_xper_feature(entry):
    if "xper" not in entry:
        return "—"
    phi = entry["xper"]["phi"]
    idx = int(np.argmax(np.abs(phi[1:]))) if len(phi) > 1 else None
    return FEATURES[idx] if idx is not None else "—"


def top_shap_feature(entry):
    if "shap_values" not in entry:
        return "—"
    sv = entry["shap_values"]
    vals = sv.values if hasattr(sv, "values") else np.asarray(sv)
    return FEATURES[int(np.argmax(np.abs(vals).mean(axis=0)))]


summary_rows = []
for name, r in results.items():
    summary_rows.append({
        "model": name,
        "AUC": round(r["auc"], 4) if "auc" in r else "—",
        "top SHAP feature": top_shap_feature(r),
        "top XPER feature": top_xper_feature(r),
        "surrogate fidelity (R²)": round(r["surrogate_fidelity"], 3) if "surrogate_fidelity" in r else "—",
    })

summary_df = pd.DataFrame(summary_rows)
summary_df

## Smoke test — remove once real models are in `models/`

A throwaway logistic regression on a small sample, just to check the pipeline runs
end to end against the real feature schema. Not a real result.

In [ ]:
from sklearn.linear_model import LogisticRegression

class _SmokeTestModule:
    _medians = None  # fixed at fit time so a later all-NaN batch (e.g. a masked coalition) still fills

    @staticmethod
    def fit(X, y):
        Xn = X.select_dtypes("number")
        _SmokeTestModule._medians = Xn.median().fillna(0)
        return LogisticRegression(max_iter=200).fit(Xn.fillna(_SmokeTestModule._medians), y)

    @staticmethod
    def predict_proba(model, X):
        Xn = X.select_dtypes("number").fillna(_SmokeTestModule._medians)
        return model.predict_proba(Xn)

if not results:
    print("Running a throwaway smoke test — NOT a real model, just checking the harness works end to end.")
    sample = train_df.sample(20_000, random_state=42)
    Xs, ys = get_X_y(sample)
    smoke_model = _SmokeTestModule.fit(Xs, ys)
    smoke_test_sample = X_test.sample(2_000, random_state=42)
    smoke_probs = _SmokeTestModule.predict_proba(smoke_model, smoke_test_sample)
    print("Smoke test predict_proba shape:", smoke_probs.shape, "— harness is wired correctly.")

    if HAS_XPER:
        # kernel-based XPER's runtime scales with feature count, not row count — cut to a
        # handful of columns and fit a dedicated small model on just those, so this stays
        # a fast plumbing check (a model fit on all features would reject a 5-column input)
        xper_cols = FEATURES[:5]
        xper_train_sample = Xs[xper_cols]
        xper_smoke_model = _SmokeTestModule.fit(xper_train_sample, ys)
        xper_test_sample = X_test.sample(50, random_state=42)[xper_cols]
        xper_smoke = compute_xper(
            xper_smoke_model, _SmokeTestModule, xper_train_sample, ys,
            xper_test_sample, y_test.loc[xper_test_sample.index],
            metric="AUC",
        )
        print("XPER smoke test phi shape:", xper_smoke["phi"].shape, "— XPER call succeeded.")